In [0]:
#display(dbutils.fs.ls("/Volumes/marathos/default/raw/"))

%restart_python

In [0]:
from utils.helpers import get_table
from pyspark.sql import functions as F
#from importlib import reload
#import transformations.bronze.ingest_bronze as bronze_module
#reload(bronze_module)
from transformations.bronze.ingest_bronze import ingest_bronze

ingest_bronze(spark)

In [0]:
df = get_table("marathos.bronze.raw_results")

In [0]:
print(f"Number of rows:    {df.count()}")
print(f"Number of columns: {len(df.columns)}")

In [0]:
#bronze layer
df.printSchema()

In [0]:
df.limit(5).display()

In [0]:
df.describe().display()

In [0]:
null_counts = df.select([
    F.count(
        F.when(F.col(c).isNull() | (F.trim(F.col(c)) == ""), c)
    ).alias(c)
    for c in df.columns
])

null_counts.display()

In [0]:
unique_events = df.select("event_name").distinct().count()
print(f"Number of unique events: {unique_events}")

In [0]:
df.select("event_name").distinct().limit(20).display()

In [0]:
df.groupBy("event_distance_raw") \
  .count() \
  .orderBy(F.desc("count")) \
  .display()

In [0]:
df.select("athlete_performance_raw") \
  .distinct() \
  .limit(20) \
  .display()

In [0]:
df.withColumn(
    "birth_year", F.col("athlete_year_of_birth").cast("double").cast("int")
).withColumn(
    "event_year", F.col("year_of_event").cast("int")
).withColumn(
    "age", F.col("event_year") - F.col("birth_year")
).groupBy("age") \
 .count() \
 .orderBy("age") \
 .filter(F.col("age").isNotNull()) \
 .display()

In [0]:
df.groupBy("athlete_country") \
  .count() \
  .orderBy(F.desc("count")) \
  .limit(20) \
  .display()

In [0]:
df.groupBy("athlete_gender") \
  .count() \
  .orderBy(F.desc("count")) \
  .display()

In [0]:
df.groupBy("year_of_event") \
  .count() \
  .orderBy("year_of_event") \
  .display()

Silver Layer after this, ran some check for bronze layer above.